# 05 — Multilingual Analysis (Module C)
**NewsBot Intelligence System 2.0 | ITAI 2373**

- C.1 Language Detection
- C.2 Translation
- C.3 Cross-Lingual Sentiment
- C.4 Coverage Comparison

In [ ]:
!pip install langdetect deep-translator vaderSentiment -q

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from langdetect import detect, detect_langs, DetectorFactory
from deep_translator import GoogleTranslator
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer
import warnings
warnings.filterwarnings('ignore')

DetectorFactory.seed = 42
vader = SentimentIntensityAnalyzer()

plt.rcParams.update({
    'figure.facecolor':'#0f1117','axes.facecolor':'#1a1d2e',
    'axes.edgecolor':'#2d3561','axes.labelcolor':'#e8d5a3',
    'xtick.color':'#a0a8c0','ytick.color':'#a0a8c0',
    'text.color':'#e8d5a3','grid.color':'#2d3561','grid.alpha':0.5,'figure.dpi':120
})
PALETTE = {'tech':'#4fc3f7','business':'#81c784','politics':'#e57373',
           'sport':'#ffb74d','entertainment':'#ce93d8'}

SUPPORTED_LANGUAGES = {
    'en':'English','fr':'French','de':'German','es':'Spanish',
    'pt':'Portuguese','it':'Italian','ar':'Arabic','zh-CN':'Chinese',
    'ja':'Japanese','ru':'Russian'
}
print('Module C setup complete.')


In [ ]:
try:
    _ = df_final
    print(f'df_final: {len(df_final)} articles')
except NameError:
    raise RuntimeError('Run notebooks 01 and 02 first.')


## C.1 — Language Detection

In [ ]:
def detect_language(text):
    try:
        probs = detect_langs(str(text)[:500])
        top   = probs[0]
        code  = str(top.lang)
        return {
            'language_code': code,
            'language_name': SUPPORTED_LANGUAGES.get(code, code.upper()),
            'confidence':    round(top.prob, 4),
        }
    except Exception as e:
        return {'language_code':'unknown','language_name':'Unknown','confidence':0.0}

# Test on sample articles
print('C.1 — Language Detection Demo')
print('='*55)
for cat in df_final.category.unique():
    sample = df_final[df_final.category==cat].sample(1, random_state=42).iloc[0]
    result = detect_language(sample.text)
    print(f'[{cat:<15}] {result["language_name"]} ({result["confidence"]*100:.0f}% confidence)')


In [ ]:
# Apply to full dataset
from tqdm import tqdm
tqdm.pandas()
print('Detecting language across full dataset...')
lang_results = df_final.text.progress_apply(detect_language)
df_final['detected_language'] = lang_results.apply(lambda r: r['language_code'])
df_final['lang_confidence']   = lang_results.apply(lambda r: r['confidence'])

print('\nLanguage distribution:')
print(df_final.detected_language.value_counts().head(10))
print(f'\nMean confidence: {df_final.lang_confidence.mean():.3f}')


## C.2 — Translation

In [ ]:
def translate_text(text, target_lang='fr', source_lang='auto', max_chars=2000):
    chunks = [text[i:i+4500] for i in range(0, min(len(text),max_chars), 4500)]
    translated = []
    for chunk in chunks:
        try:
            t = GoogleTranslator(source=source_lang, target=target_lang).translate(chunk)
            translated.append(t)
        except Exception as e:
            translated.append(chunk)
    return {
        'translated_text': ' '.join(translated),
        'target_lang':     target_lang,
        'char_count':      len(' '.join(translated))
    }

# Demo translation
print('C.2 — Translation Demo')
print('='*55)
sample = df_final[df_final.category=='tech'].sample(1, random_state=42).iloc[0]
print(f'Original (EN): {sample.text[:200]}...\n')

for lang_code, lang_name in [('fr','French'),('de','German'),('es','Spanish')]:
    result = translate_text(sample.text, target_lang=lang_code)
    print(f'[{lang_name}]: {result["translated_text"][:150]}...')
    print()


## C.3 — Cross-Lingual Sentiment Analysis

In [ ]:
def cross_lingual_sentiment(article_text, languages=None):
    """Analyze sentiment after translating to multiple languages."""
    targets  = languages or ['fr','de','es','it']
    results  = {}

    # English baseline
    scores = vader.polarity_scores(article_text[:2000])
    c = scores['compound']
    results['en'] = {
        'compound': c,
        'label': 'Positive' if c>=0.05 else 'Negative' if c<=-0.05 else 'Neutral'
    }

    for lang in targets:
        try:
            fwd   = translate_text(article_text, target_lang=lang)
            back  = translate_text(fwd['translated_text'], target_lang='en', source_lang=lang)
            scores = vader.polarity_scores(back['translated_text'])
            c = scores['compound']
            results[lang] = {
                'compound':           c,
                'label':              'Positive' if c>=0.05 else 'Negative' if c<=-0.05 else 'Neutral',
                'translated_preview': fwd['translated_text'][:100]
            }
        except Exception as e:
            results[lang] = {'error': str(e)}
    return results

# Demo
print('C.3 — Cross-Lingual Sentiment Demo')
print('='*55)
sample = df_final[df_final.category=='politics'].sample(1, random_state=7).iloc[0]
print(f'Article: {sample.text[:200]}...\n')

cl_results = cross_lingual_sentiment(sample.text)
print('Sentiment by language (after round-trip translation):')
for lang, data in cl_results.items():
    if 'compound' in data:
        lang_name = SUPPORTED_LANGUAGES.get(lang, lang.upper())
        print(f'  {lang_name:<12} compound: {data["compound"]:+.3f}  ({data["label"]})')


In [ ]:
# Visualize cross-lingual sentiment shifts
langs   = [l for l,d in cl_results.items() if 'compound' in d]
scores  = [cl_results[l]['compound'] for l in langs]
lang_names = [SUPPORTED_LANGUAGES.get(l, l.upper()) for l in langs]
colors  = ['#81c784' if s>=0.05 else '#e57373' if s<=-0.05 else '#e8d5a3' for s in scores]

fig, ax = plt.subplots(figsize=(10,4))
bars = ax.bar(lang_names, scores, color=colors, alpha=0.85, width=0.5)
ax.axhline(0, color='#a0a8c0', linewidth=1, linestyle='--')
ax.axhspan(0.05, 1.0, alpha=0.05, color='#81c784')
ax.axhspan(-1.0, -0.05, alpha=0.05, color='#e57373')
ax.set_title('Cross-Lingual Sentiment — Round-Trip Translation', fontsize=12, fontweight='bold')
ax.set_ylabel('VADER Compound Score'); ax.set_ylim(-1,1)
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig('cross_lingual_sentiment.png',bbox_inches='tight',dpi=150,facecolor='#0f1117')
plt.show()


## C.4 — Coverage Comparison Across Categories

In [ ]:
def compare_coverage(df, keyword, text_col='cleaned_text' if 'cleaned_text' in dir() else 'text'):
    """Compare how a keyword topic is covered across categories."""
    col  = 'cleaned_text' if 'cleaned_text' in df.columns else 'text'
    mask = df[col].str.lower().str.contains(keyword.lower(), na=False)
    sub  = df[mask].copy()
    if len(sub) == 0:
        print(f'No articles found for keyword: {keyword}')
        return pd.DataFrame()
    if 'sentiment_compound' not in sub.columns:
        sents = sub[col].apply(lambda t: vader.polarity_scores(t[:2000])['compound'])
        sub['sentiment_compound'] = sents
    return sub.groupby('category').agg(
        article_count=('sentiment_compound','count'),
        avg_sentiment=('sentiment_compound','mean')
    ).sort_values('article_count', ascending=False)

# Test with a few keywords
print('C.4 — Coverage Comparison')
print('='*55)
for keyword in ['technology','government','market']:
    result = compare_coverage(df_final, keyword)
    if len(result) > 0:
        print(f'\nKeyword: "{keyword}" ({result.article_count.sum()} articles)')
        print(result.to_string())


In [ ]:
# Heatmap: topic coverage × category
keywords = ['technology','economy','election','championship','film','climate','bank','police']
heatmap_data = []
col = 'cleaned_text' if 'cleaned_text' in df_final.columns else 'text'
cats = sorted(df_final.category.unique())

for kw in keywords:
    row = {}
    for cat in cats:
        sub   = df_final[df_final.category==cat]
        count = sub[col].str.lower().str.contains(kw, na=False).sum()
        row[cat] = count / len(sub)
    heatmap_data.append(row)

heat_df = pd.DataFrame(heatmap_data, index=keywords)

fig, ax = plt.subplots(figsize=(12,6))
sns.heatmap(heat_df, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax,
            annot_kws={'size':9}, linewidths=0.5)
ax.set_title('Topic Keyword Coverage by Category (proportion)', fontsize=12, fontweight='bold')
ax.set_xlabel('Category'); ax.set_ylabel('Keyword')
plt.tight_layout()
plt.savefig('coverage_heatmap.png',bbox_inches='tight',dpi=150,facecolor='#0f1117')
plt.show()


In [ ]:
print('='*55)
print('  MODULE C SUMMARY')
print('='*55)
en_articles = (df_final.detected_language=='en').sum()
print(f'  Language detection   : {len(df_final)} articles processed')
print(f'  English articles     : {en_articles} ({en_articles/len(df_final)*100:.1f}%)')
print(f'  Supported languages  : {len(SUPPORTED_LANGUAGES)}')
print(f'  Translation engine   : Google Translate (deep-translator)')
print(f'  Sentiment analysis   : VADER (cross-lingual round-trip)')
print('='*55)
